In [6]:
import sys
import torch
import transformers
import datasets

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)

device = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

Python: 3.11.15 (main, Mar  3 2026, 00:52:57) [Clang 21.0.0 (clang-2100.0.123.102)]
PyTorch: 2.12.0
Transformers: 5.9.0
Datasets: 4.8.5
Device: mps


In [7]:
from datasets import load_dataset

python_ds = load_dataset(
    "code_search_net",
    "python"
)

In [8]:
print(python_ds)

DatasetDict({
    train: Dataset({
        features: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url'],
        num_rows: 412178
    })
    test: Dataset({
        features: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url'],
        num_rows: 22176
    })
    validation: Dataset({
        features: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url'],
        num_rows: 23107
    })
})


In [9]:
python_ds["train"].column_names

['repository_name',
 'func_path_in_repository',
 'func_name',
 'whole_func_string',
 'language',
 'func_code_string',
 'func_code_tokens',
 'func_documentation_string',
 'func_documentation_tokens',
 'split_name',
 'func_code_url']

In [10]:
sample = python_ds["train"][0]

print("Repository:", sample["repository_name"])
print("Path:", sample["func_path_in_repository"])
print("Function:", sample["func_name"])
print("\nDocstring / Query:")
print(sample["func_documentation_string"])
print("\nCode / Retrieval Target:")
print(sample["func_code_string"])

Repository: mjirik/imcut
Path: imcut/pycut.py
Function: ImageGraphCut.__msgc_step3_discontinuity_localization

Docstring / Query:
Estimate discontinuity in basis of low resolution image segmentation.
        :return: discontinuity in low resolution

Code / Retrieval Target:
def __msgc_step3_discontinuity_localization(self):
        """
        Estimate discontinuity in basis of low resolution image segmentation.
        :return: discontinuity in low resolution
        """
        import scipy

        start = self._start_time
        seg = 1 - self.segmentation.astype(np.int8)
        self.stats["low level object voxels"] = np.sum(seg)
        self.stats["low level image voxels"] = np.prod(seg.shape)
        # in seg is now stored low resolution segmentation
        # back to normal parameters
        # step 2: discontinuity localization
        # self.segparams = sparams_hi
        seg_border = scipy.ndimage.filters.laplace(seg, mode="constant")
        logger.debug("seg_border: %s", 

<!-- Create working dataset -->

In [11]:
small_train = python_ds["train"].select(range(1000))

print(small_train)

Dataset({
    features: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url'],
    num_rows: 1000
})


In [12]:
queries = small_train["func_documentation_string"]
codes = small_train["func_code_string"]
names = small_train["func_name"]

print("Queries:", len(queries))
print("Codes:", len(codes))

Queries: 1000
Codes: 1000


In [13]:
for i in range(3):
    print("=" * 80)

    print("Function:")
    print(names[i])

    print("\nQuery:")
    print(queries[i])

    print("\nCode:")
    print(codes[i][:300])

    print("\n")

Function:
ImageGraphCut.__msgc_step3_discontinuity_localization

Query:
Estimate discontinuity in basis of low resolution image segmentation.
        :return: discontinuity in low resolution

Code:
def __msgc_step3_discontinuity_localization(self):
        """
        Estimate discontinuity in basis of low resolution image segmentation.
        :return: discontinuity in low resolution
        """
        import scipy

        start = self._start_time
        seg = 1 - self.segmentation.astype(


Function:
ImageGraphCut.__multiscale_gc_lo2hi_run

Query:
Run Graph-Cut segmentation with refinement of low resolution multiscale graph.
        In first step is performed normal GC on low resolution data
        Second step construct finer grid on edges of segmentation from first
        step.
        There is no option for use without `use_boundary_penalties`

Code:
def __multiscale_gc_lo2hi_run(self):  # , pyed):
        """
        Run Graph-Cut segmentation with refinement of low resolutio

<!-- Load CodeBERT -->

In [14]:
from transformers import AutoTokenizer, AutoModel
import torch

In [15]:
# load model
model_name = "microsoft/codebert-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

print("CodeBERT loaded successfully")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CodeBERT loaded successfully


In [16]:
# understand the tokenizer
sample_query = queries[0]

tokens = tokenizer(
    sample_query,
    return_tensors="pt",
    truncation=True,
    max_length=256
)

print(tokens.keys())
print(tokens["input_ids"].shape)

KeysView({'input_ids': tensor([[    0, 27602, 16633, 29649, 15147,    11,  1453,     9,   614,  3547,
          2274,  2835,  1258,     4, 50118,  1437,  1437,  1437,  1437,  1437,
          1437,  1437,  4832, 30921,    35, 29649, 15147,    11,   614,  3547,
             2]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1]])})
torch.Size([1, 31])


In [17]:
# first embedding
with torch.no_grad():
    outputs = model(**tokens)

print(outputs.last_hidden_state.shape)

torch.Size([1, 31, 768])


<!-- here the 768 means the CodeBERT converts text into a 768-dimensional semantic vector space -->

In [18]:
# extract the [CLS] token embedding
query_embedding = outputs.last_hidden_state[:, 0, :]

print(query_embedding.shape)

torch.Size([1, 768])


In [19]:
# passing tokens through CodeBERT
with torch.no_grad():
    outputs = model(**tokens)
    
print(outputs.last_hidden_state.shape)

torch.Size([1, 31, 768])


In [20]:
# extract query embedding
query_embedding = outputs.last_hidden_state[:, 0, :]
print(query_embedding.shape)

torch.Size([1, 768])


In [21]:
# view 1st few values of the query embedding
print(query_embedding[0][:10])

tensor([-0.0349,  0.2808, -0.0093,  0.0742,  0.1204, -0.2885,  0.0535,  0.0431,
         0.3062, -0.3289])


In [22]:
print(query_embedding.shape)

torch.Size([1, 768])


In [23]:
print("queries" in globals())
print("model" in globals())
print("tokenizer" in globals())

True
True
True


In [24]:
sample_query = queries[0]

tokens = tokenizer(
    sample_query,
    return_tensors="pt",
    truncation=True,
    max_length=256
)

with torch.no_grad():
    outputs = model(**tokens)

query_embedding = outputs.last_hidden_state[:, 0, :]

print(query_embedding.shape)

torch.Size([1, 768])


In [25]:
def get_embedding(text):
    tokens = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    with torch.no_grad():
        outputs = model(**tokens)

    return outputs.last_hidden_state[:, 0, :]

In [26]:
test_embedding = get_embedding("read csv file")
print(test_embedding.shape)

torch.Size([1, 768])


In [27]:
code_embedding = get_embedding(codes[0])

print(code_embedding.shape)

torch.Size([1, 768])


In [28]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(
    test_embedding.numpy(),
    code_embedding.numpy()
)

print(similarity)

[[0.9385775]]


In [29]:
code_embeddings = []

for code in codes[:100]:
    embedding = get_embedding(code)
    code_embeddings.append(embedding)

print("Generated", len(code_embeddings), "embeddings")

Generated 100 embeddings
